# [LELEC2870] Machine Learning Project — Heart failure in the Smurf society
# Part 2 — Nonlinear Models

---

This notebook uses Part 1 as a baseline to beat.

**Constraints** (from the project guidelines):
- at most 6 features from the original tabular dataset;
- evaluation criterion = RW-RMSE, with weights `w = 1 + y/0.1`;
- `X_test.csv` / `y_test.csv` are the test set: used once, at the very end

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression

from sklearn.feature_selection import mutual_info_regression, SequentialFeatureSelector
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup OK")

Setup OK


## 1. Data loading

We reload the raw CSV files

Before any modelling we verify a set of explicit assumptions just to check if every data has its correspondance label around the columns. Each assumption is written as an assert, so the notebook stops immediately if a later edit breaks one, instead of silently producing a wrong number.

In [2]:
DATA = "data_student/data_labeled/"

X_train_raw = pd.read_csv(DATA + "X_train.csv")
X_test_raw  = pd.read_csv(DATA + "X_test.csv")
y_train = pd.read_csv(DATA + "y_train.csv", header=None).values.ravel()
y_test  = pd.read_csv(DATA + "y_test.csv",  header=None).values.ravel()

# `img_filename` points to the heart scan used in Part 3. Part 2 is tabular only.
IMG_COL = "img_filename"
X_train_raw = X_train_raw.drop(columns=[IMG_COL])
X_test_raw  = X_test_raw.drop(columns=[IMG_COL])

# The 13 original tabular features, split by type.
CATEGORICAL = ["profession", "sarsaparilla", "smurfberry_liquor", "smurfin_donuts"]
NUMERIC     = [c for c in X_train_raw.columns if c not in CATEGORICAL]

# The three consumption variables are ordered scales, not unordered labels.
ORDINAL     = ["sarsaparilla", "smurfberry_liquor", "smurfin_donuts"]
NOMINAL     = ["profession"]
ORD_LEVELS  = ["very_low", "low", "moderate", "high", "very_high"]

print(f"X_train {X_train_raw.shape}   X_test {X_test_raw.shape}")
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = "
      f"{len(NUMERIC)+len(CATEGORICAL)} original features")
print(f"numeric    : {NUMERIC}")
print(f"categorical: {CATEGORICAL}")

X_train (1000, 13)   X_test (500, 13)
9 numeric + 4 categorical = 13 original features
numeric    : ['age', 'blood_pressure', 'calcium', 'cholesterol', 'hemoglobin', 'height', 'potassium', 'vitamin_D', 'weight']
categorical: ['profession', 'sarsaparilla', 'smurfberry_liquor', 'smurfin_donuts']


Six assumptions, each one a thing that would silently corrupt a result if it were false.

In [3]:
# (1) Row counts match their labels.
assert len(X_train_raw) == len(y_train) == 1000
assert len(X_test_raw)  == len(y_test)  == 500

# (2) No missing values anywhere -> no imputation step is needed.
assert X_train_raw.isna().sum().sum() == 0
assert X_test_raw.isna().sum().sum()  == 0

# (3) Train and test carry exactly the same columns, in the same order.
assert list(X_train_raw.columns) == list(X_test_raw.columns)

# (4) The target is a probability. This is what justifies clipping predictions
#     to [0, 1] later, and it is what makes the RW-RMSE weights bounded.
assert y_train.min() >= 0.0 and y_train.max() <= 1.0
assert y_test.min()  >= 0.0 and y_test.max()  <= 1.0

# (5) No categorical level appears in test that was never seen in training.
for c in CATEGORICAL:
    unseen = set(X_test_raw[c]) - set(X_train_raw[c])
    assert not unseen, f"{c}: unseen levels in test -> {unseen}"

# (6) The three consumption variables use exactly the five expected levels.
for c in ORDINAL:
    assert set(X_train_raw[c]) == set(ORD_LEVELS), f"{c}: unexpected levels"

print("Data contract: all 6 assumptions hold.")
print(f"  y_train in [{y_train.min():.4f}, {y_train.max():.4f}]   mean {y_train.mean():.4f}")
print(f"  y_test  in [{y_test.min():.4f}, {y_test.max():.4f}]   mean {y_test.mean():.4f}")
print(f"  duplicate training rows: {X_train_raw.duplicated().sum()}")
print(f"  profession levels ({X_train_raw['profession'].nunique()}): "
      f"{sorted(X_train_raw['profession'].unique())}")

Data contract: all 6 assumptions hold.
  y_train in [0.0000, 0.4900]   mean 0.0807
  y_test  in [0.0000, 0.4900]   mean 0.0850
  duplicate training rows: 0
  profession levels (6): ['administration_and_governance', 'craftsmanship', 'food_production', 'manufacturing', 'resource_extraction', 'services']


## 1.2 Extrapolation check — specific to Part 2

A linear model extrapolates: outside the training range it keeps going in a straight
line. The nonlinear models of Part 2 behave very differently, and we have two classes that behave differently from each other:

- **tree-based models (tree, forest, boosting) and k-NN go completely flat** outside
  the training range. They can only reproduce values they have seen, so for a Smurf
  beyond the edge of the data they return the value of the nearest edge.
- **SVR and MLP keep moving**, but in a direction that nothing in the data constrains.

So we check how many test Smurfs fall outside the training range on at least one
numeric feature. It is just a limitation to to know about when the final score is computed.

In [4]:
rows = []
for c in NUMERIC:
    lo, hi = X_train_raw[c].min(), X_train_raw[c].max()
    n_out = int(((X_test_raw[c] < lo) | (X_test_raw[c] > hi)).sum())
    rows.append({"feature": c,
                 "train_min": round(lo, 2),  "train_max": round(hi, 2),
                 "test_min": round(X_test_raw[c].min(), 2),
                 "test_max": round(X_test_raw[c].max(), 2),
                 "test_outside": n_out})

extrap = pd.DataFrame(rows).set_index("feature")
outside_any = np.any([((X_test_raw[c] < X_train_raw[c].min()) |
                       (X_test_raw[c] > X_train_raw[c].max())).values
                      for c in NUMERIC], axis=0)

print(extrap.to_string())
print(f"\nTest Smurfs outside the training range on >=1 feature: "
      f"{outside_any.sum()} / {len(X_test_raw)} ({100*outside_any.mean():.1f}%)")

                train_min  train_max  test_min  test_max  test_outside
feature                                                               
age                 26.00     180.00     32.00    180.00             0
blood_pressure      88.03     163.62     84.37    154.44             1
calcium              1.78       3.11      1.88      3.02             0
cholesterol         39.37     189.06     50.72    188.79             0
hemoglobin           8.02      20.04      7.69     20.39             3
height               6.36       8.93      6.52      8.75             0
potassium            2.78       5.57      2.68      5.89             3
vitamin_D           11.39      48.61     15.27     47.52             0
weight              60.13     151.35     63.95    153.36             2

Test Smurfs outside the training range on >=1 feature: 9 / 500 (1.8%)


## 2. Preprocessing rebuilt for nonlinear models

Part 1 compared four models — OLS, WLS, Ridge and Lasso — but all four belonged to a
single family: every one of them solved a linear system, so a single preprocessing
pipeline served all four.

Part 2 compares six models spread across **two families**, and the correct
preprocessing is **not the same for both**:

| family | sees a feature as | standardisation |
|---|---|---|
| **distance-based** — k-NN, SVR, MLP | a position in space | **required** |
| **split-based** — tree, random forest, boosting | an ordering of values | **irrelevant** |

So `make_preprocessor` takes a `scale` flag. Everything else — the ordinal encoding
of the three consumption scales, the one-hot encoding of `profession` — is shared.

All encoders are fitted **inside** a `Pipeline`, so that during cross-validation they
are fitted on the training folds only. This is what prevents the validation fold from
leaking into the scaler's mean and standard deviation.


In [ ]:
def make_preprocessor(features, scale):

    #features : list of original column names (at most 6, per the guidelines)
    #scale    : True  -> standardise numeric and ordinal columns (distance-based models)
               #False -> leave them in their original units (split-based models)
    f_num = [c for c in features if c in NUMERIC]
    f_ord = [c for c in features if c in ORDINAL]
    f_nom = [c for c in features if c in NOMINAL]

    blocks = []

    if f_num:
        blocks.append(("num", StandardScaler() if scale else "passthrough", f_num))

    if f_ord:
        # The five levels are an ordered scale, so we map them to 0..4 and keep that
        # order. A split-based model uses only the order; a distance-based model also
        # needs the result standardised, like any other numeric column.
        steps = [("encode", OrdinalEncoder(categories=[ORD_LEVELS] * len(f_ord)))]
        if scale:
            steps.append(("scale", StandardScaler()))
        blocks.append(("ord", Pipeline(steps), f_ord))

    if f_nom:
        # `profession` has no natural order, so it is one-hot encoded.
        # drop="first" removes the redundant column (dummy variable trap).
        blocks.append(("nom", OneHotEncoder(drop="first", sparse_output=False), f_nom))

    return ColumnTransformer(blocks, remainder="drop")


# Which models need scaling. Used everywhere below.
NEEDS_SCALING = {"knn": True, "svr": True, "mlp": True,
                 "tree": False, "forest": False, "boosting": False}

print("Counting rule (guidelines footnote): one-hot encoding `profession` produces")
print("5 columns but still counts as 1 of the 6 allowed original features.")

make_preprocessor ready.
Counting rule (guidelines footnote): one-hot encoding `profession` produces
5 columns but still counts as 1 of the 6 allowed original features.


### 2.1 Evidence for the two-pipeline decision

Below we verify the claim rather than assume it. Each model is cross-validated twice
on all 13 features — once on raw units, once standardised. Nothing else changes.

In [8]:
def quick_cv(model, scale, features, sample_weight=True):
    #5-fold CV RW-RMSE. Preprocessing is refitted inside every fold
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr, va in kf.split(X_train_raw):
        pipe = Pipeline([("prep", make_preprocessor(features, scale)),
                         ("model", model)])
        if sample_weight:
            pipe.fit(X_train_raw.iloc[tr], y_train[tr],
                     model__sample_weight=1.0 + y_train[tr] / 0.1)
        else:
            pipe.fit(X_train_raw.iloc[tr], y_train[tr])
        pred = pipe.predict(X_train_raw.iloc[va])
        w = 1.0 + y_train[va] / 0.1
        scores.append(np.sqrt(np.sum(w * (y_train[va] - pred) ** 2) / np.sum(w)))
    return float(np.mean(scores))


ALL_FEATURES = NUMERIC + CATEGORICAL

tests = [
    ("k-NN (k=25)",         KNeighborsRegressor(n_neighbors=25),                     False),
    ("SVR (RBF, gamma=.05)", SVR(C=1.0, gamma=0.05),                                 False),
    ("Decision tree (d=5)", DecisionTreeRegressor(max_depth=5, random_state=0),      True),
    ("Random forest",       RandomForestRegressor(n_estimators=300, min_samples_leaf=10,
                                                  random_state=0, n_jobs=1),         True),
    ("Gradient boosting",   HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                                          max_leaf_nodes=15, min_samples_leaf=30,
                                                          random_state=0),           True),
]

print(f"{'model':24s} {'raw units':>11s} {'standardised':>14s} {'gain':>9s}")
print("-" * 62)
for name, mdl, use_w in tests:
    raw = quick_cv(mdl, scale=False, features=ALL_FEATURES, sample_weight=use_w)
    std = quick_cv(mdl, scale=True,  features=ALL_FEATURES, sample_weight=use_w)
    print(f"{name:24s} {raw:11.5f} {std:14.5f} {100*(raw-std)/raw:+8.1f}%")

model                      raw units   standardised      gain
--------------------------------------------------------------
k-NN (k=25)                  0.07351        0.06948     +5.5%
SVR (RBF, gamma=.05)         0.09904        0.06114    +38.3%
Decision tree (d=5)          0.06747        0.06747     -0.0%
Random forest                0.05719        0.05718     +0.0%
Gradient boosting            0.05290        0.05289     +0.0%


### 2.2 Encoding decision for the three consumption scales

`sarsaparilla`, `smurfberry_liquor` and `smurfin_donuts` are **ordered** scales
(`very_low` < `low` < `moderate` < `high` < `very_high`), so they are mapped to
0..4 with `OrdinalEncoder`. This keeps the ordering, and costs one column instead
of five.

For the split-based models this encoding carries **no hidden assumption**: a tree
never multiplies the value, it only asks "is this Smurf above level 2?", so the
numbers supply nothing but the ordering and the model is free to place its own
thresholds. (In Part 1 the same encoding did impose an assumption on the linear
model, namely that consecutive levels are equally spaced.)

`profession` is **unordered** — no profession is "more" than another — so it is the
only variable that is one-hot encoded.